In [2]:
!pip install fuzzywuzzy

In [3]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd
import numpy as np
from collections import OrderedDict
import scipy.stats as stats

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
def extractLastNames(names):
    lastNames = []
    for name in names:
      parts = name.strip().lower().split()
      lastNames.append(parts[-1].strip())
    return lastNames

In [5]:
thresholds = [60, 70, 80, 90]

In [6]:
df = pd.read_csv("researchers.csv")
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Region,Co-author count (Google Scholar),Co-authors' names (Google Scholar),Co-authors' names (OpenAlex),Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral)
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,East/South-East Asia,3,K Ranjeet/Neelesh Dahanukar/Rajeev Raghavan,B Madhusoodana Kurup/Balakrishna Madhusoodana ...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,East/South-East Asia,3,Jamroon Srichaichana/Dokrak Marod/Hee Han,Akbar I. Inamdar/Arjun Magotra/Atchara Teerawa...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,East/South-East Asia,3,A.R.S.B. Athauda/Saman Athauda/M Pagthinathan/...,A. R. S. B. Athauda/Clement R. de Cruz/E. Pows...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,East/South-East Asia,3,Poulomi Dey/Jiwan Paudel/Ashish Chaudhary,Akshay Chaudhary/Alberto Sanz-Cobeña/Alice Min...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,East/South-East Asia,5,S. Venkatesan/T.Uma Maheswari/S Kumar/R. Sudha...,A. Bedoui/A. Deepak/A. K. Ramasamy/A. Kanakala...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-


In [7]:
df = df[df["Reality (Llama 4 Scout)"] != "hypothetical"]
df = df[df["Co-authors' names (Llama 4 Scout)"].notna()]
df = df[df["Co-authors' names (OpenAlex)"].notna()]

In [8]:
len(df)

512

In [9]:
df["Match Count 60%"] = 0
df["Match Count 70%"] = 0
df["Match Count 80%"] = 0
df["Match Count 90%"] = 0

In [ ]:
for index, row in df.iterrows():
    llmNames = row["Co-authors' names (Llama 4 Scout)"].split('/')
    refNames = row["Co-authors' names (OpenAlex)"].split('/')

    llmNames = extractLastNames(llmNames)
    refNames = extractLastNames(refNames)

    for threshold in thresholds:
        count = 0
        refNamesCopy = refNames.copy()

        for name in llmNames:
            if not refNamesCopy:
                break

            bestRatio = 0
            bestIndex = -1
            for i, candidate in enumerate(refNamesCopy):
                currentRatio = fuzz.ratio(name, candidate)
                if currentRatio >= threshold and currentRatio > bestRatio:
                    bestRatio = currentRatio
                    bestIndex = i

            if bestIndex != -1:
                count += 1
                del refNamesCopy[bestIndex]

        colName = "Match Count " + str(threshold) + "%"
        df.at[index, colName] = count

In [ ]:
df.head()

In [ ]:
df["DNE 60%"] = 0
df["DNE 70%"] = 0
df["DNE 80%"] = 0
df["DNE 90%"] = 0

for index, row in df.iterrows():

    N = len(row["Co-authors' names (Google Scholar)"].split("/"))
    R = len(row["Co-authors' names (OpenAlex)"].split("/"))

    for threshold in [60, 70, 80, 90]:
        match_col = f"Match Count {threshold}%"
        DNE_col = f"DNE {threshold}%"

        denom = max(1, min(N, R))
        DNE = min(row[match_col] / denom, 1.0)

        df.at[index, DNE_col] = DNE

In [ ]:
print("Mean DNE", np.mean(df["DNE 60%"]))
print("Std DNE", df["DNE 60%"].std())

In [ ]:
fields = list(df["Field"].unique())
fields

In [ ]:
metrics = ["DNE 60%", "DNE 70%", "DNE 80%", "DNE 90%"]

levels = ['High', 'Low']

In [16]:
regions = list(df["Region"].unique())
regions

['East/South-East Asia',
 'Europe',
 'Middle East',
 'North Africa',
 'North America',
 'Oceanic',
 'South/Central America',
 'Sub-Saharan Africa']

# Mean DNE without disaggregation

In [17]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
      print(metric, "*****", sf[metric].mean())

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.47531293298444066
DNE 70% ***** 0.2669767883431994
DNE 80% ***** 0.1893662826303814
DNE 90% ***** 0.1494340947283904
Biology
DNE 60% ***** 0.5012128769188893
DNE 70% ***** 0.29048451284665533
DNE 80% ***** 0.20613819979065215
DNE 90% ***** 0.15909461334134892
Built Environment & Design
DNE 60% ***** 0.45092249044035543
DNE 70% ***** 0.22575357606833568
DNE 80% ***** 0.1616532267806721
DNE 90% ***** 0.11644319375967453
Clinical Medicine
DNE 60% ***** 0.5362867273928046
DNE 70% ***** 0.3330103815505525
DNE 80% ***** 0.19487915831797978
DNE 90% ***** 0.12617067540688745
Earth & Environmental Sciences
DNE 60% ***** 0.49153740632479
DNE 70% ***** 0.2604099026376646
DNE 80% ***** 0.17932336502126028
DNE 90% ***** 0.1440471596369457
Economics & Business
DNE 60% ***** 0.31876731932631713
DNE 70% ***** 0.13700135601635927
DNE 80% ***** 0.08900381689744867
DNE 90% ***** 0.07422485748378604
Engineering
DNE 60% ***** 0.44136767763715734
DNE 70% ***

In [18]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    print(metric, "*****", sf[metric].mean())

East/South-East Asia
DNE 60% ***** 0.6037992880873075
DNE 70% ***** 0.4683584744339926
DNE 80% ***** 0.3953991862910667
DNE 90% ***** 0.3433194488389486
Europe
DNE 60% ***** 0.3787953019175616
DNE 70% ***** 0.18228451862716208
DNE 80% ***** 0.12402708211027794
DNE 90% ***** 0.07622697782439254
Middle East
DNE 60% ***** 0.47068338216088795
DNE 70% ***** 0.25680403519392153
DNE 80% ***** 0.13937495695168364
DNE 90% ***** 0.08098447988059669
North Africa
DNE 60% ***** 0.3971406342564016
DNE 70% ***** 0.15332038540008805
DNE 80% ***** 0.09190085240928965
DNE 90% ***** 0.060978117216257635
North America
DNE 60% ***** 0.4802249310212044
DNE 70% ***** 0.2784252643526003
DNE 80% ***** 0.21023911500894768
DNE 90% ***** 0.1765936825023828
Oceanic
DNE 60% ***** 0.4742330318169435
DNE 70% ***** 0.2456153805967945
DNE 80% ***** 0.1846060383227926
DNE 90% ***** 0.1429620838163086
South/Central America
DNE 60% ***** 0.3745454390265533
DNE 70% ***** 0.16258809139194236
DNE 80% ***** 0.0957577916239197

# Mean DNE disaggregated by Citation Level

In [19]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

Agriculture, Fisheries & Forestry
High
DNE 60% ***** 0.5165767831104737
DNE 70% ***** 0.32627953504199836
DNE 80% ***** 0.2577663532397971
DNE 90% ***** 0.2066710137852105
Low
DNE 60% ***** 0.40825917652963684
DNE 70% ***** 0.17060982495765104
DNE 80% ***** 0.07821616789008093
DNE 90% ***** 0.05642410126105778
Biology
High
DNE 60% ***** 0.5719090098733101
DNE 70% ***** 0.35410312684098966
DNE 80% ***** 0.2608918638773152
DNE 90% ***** 0.20007942285748423
Low
DNE 60% ***** 0.2596677559912854
DNE 70% ***** 0.07312091503267974
DNE 80% ***** 0.01906318082788671
DNE 90% ***** 0.01906318082788671
Built Environment & Design
High
DNE 60% ***** 0.4755199961944117
DNE 70% ***** 0.25852988729339216
DNE 80% ***** 0.1709260766002519
DNE 90% ***** 0.12858347591599964
Low
DNE 60% ***** 0.38383838383838376
DNE 70% ***** 0.13636363636363635
DNE 80% ***** 0.13636363636363635
DNE 90% ***** 0.08333333333333333
Clinical Medicine
High
DNE 60% ***** 0.6382130288012641
DNE 70% ***** 0.4126159651894946
DNE 80%

In [20]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

East/South-East Asia
High
DNE 60% ***** 0.687606141708427
DNE 70% ***** 0.5618122379509347
DNE 80% ***** 0.47318448126554774
DNE 90% ***** 0.411141913753457
Low
DNE 60% ***** 0.2825396825396826
DNE 70% ***** 0.11011904761904762
DNE 80% ***** 0.09722222222222222
DNE 90% ***** 0.08333333333333333
Europe
High
DNE 60% ***** 0.41052794874974596
DNE 70% ***** 0.19749903988030273
DNE 80% ***** 0.1361558995724125
DNE 90% ***** 0.07634448675397747
Low
DNE 60% ***** 0.30390625539360644
DNE 70% ***** 0.1463782484697501
DNE 80% ***** 0.09540307289964041
DNE 90% ***** 0.07594965675057208
Middle East
High
DNE 60% ***** 0.5284939370101759
DNE 70% ***** 0.30718280371067297
DNE 80% ***** 0.189222537473284
DNE 90% ***** 0.10556407042159517
Low
DNE 60% ***** 0.3801972963098283
DNE 70% ***** 0.1779503105590062
DNE 80% ***** 0.061352657004830925
DNE 90% ***** 0.042512077294685986
North Africa
High
DNE 60% ***** 0.4242711686563964
DNE 70% ***** 0.22008367891120317
DNE 80% ***** 0.13531975213325173
DNE 90% *

# Mean Comparison

In [21]:
lowCitedDNE = df[df["Citation Level"] == "Low"]["DNE 60%"].to_list()
highlyCitedDNE = df[df["Citation Level"] == "High"]["DNE 60%"].to_list()

In [22]:
print("Low cited Avg DNE:", np.mean(lowCitedDNE))
print("highly cited Avg DNE:", np.mean(highlyCitedDNE))

Low cited Avg DNE: 0.31716521201635106
highly cited Avg DNE: 0.49077132530083306


In [23]:
t_stat, p_value = stats.ttest_ind(highlyCitedDNE, lowCitedDNE, alternative='greater')

In [24]:
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

T-statistic: 5.5082859831470214
P-value: 2.877492191802293e-08


## T-Test

In [25]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.15779273898558
DNE 70% ***** 0.03895572191043738
DNE 80% ***** 0.014309322280289062
DNE 90% ***** 0.02413044644975234
Biology
DNE 60% ***** 0.0023552700193801897
DNE 70% ***** 0.0012192756321187184
DNE 80% ***** 0.0021240255053598333
DNE 90% ***** 0.011533844994821836
Built Environment & Design
DNE 60% ***** 0.21856087358026122
DNE 70% ***** 0.10693975454579611
DNE 80% ***** 0.34028654676151515
DNE 90% ***** 0.2678858074471772
Clinical Medicine
DNE 60% ***** 0.003931321386230234
DNE 70% ***** 0.016080572336961266
DNE 80% ***** 0.02081245080975825
DNE 90% ***** 0.03300576110308995
Earth & Environmental Sciences
DNE 60% ***** 0.00025501703570110577
DNE 70% ***** 0.009888598368804732
DNE 80% ***** 0.08970704942069245
DNE 90% ***** 0.15669514873114826
Economics & Business
DNE 60% ***** 0.14320518055763057
DNE 70% ***** 0.018745257459952397
DNE 80% ***** 0.08659592662902603
DNE 90% ***** 0.03554627193683816
Engineering
DNE 60% ***** 0.009515

In [26]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 8.346957996341635e-05
DNE 70% ***** 8.136274617518849e-05
DNE 80% ***** 0.0007996180892516327
DNE 90% ***** 0.002629701800697412
Europe
DNE 60% ***** 0.08665717501146744
DNE 70% ***** 0.18263654659304068
DNE 80% ***** 0.2059993064286888
DNE 90% ***** 0.49580435879970375
Middle East
DNE 60% ***** 0.049529435422778614
DNE 70% ***** 0.03032514649819297
DNE 80% ***** 0.00827304153332747
DNE 90% ***** 0.03396689164913309
North Africa
DNE 60% ***** 0.23562521435838907
DNE 70% ***** 0.0024957532467685876
DNE 80% ***** 0.011457478958190558
DNE 90% ***** 0.06130373810646067
North America
DNE 60% ***** 0.03982619055153416
DNE 70% ***** 0.08708075628553229
DNE 80% ***** 0.13377823549734094
DNE 90% ***** 0.11923288424082272
Oceanic
DNE 60% ***** 0.05703816713813225
DNE 70% ***** 0.019645597465650796
DNE 80% ***** 0.1542556261401917
DNE 90% ***** 0.08911737634549756
South/Central America
DNE 60% ***** 0.013962429633546072
DNE 70% ***** 0.021673906872448998
DNE 80%

## Mann-Whitney U Test

In [27]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.16832219275937754
DNE 70% ***** 0.02331769521808353
DNE 80% ***** 0.0027771807232502056
DNE 90% ***** 0.010469059001444188
Biology
DNE 60% ***** 0.0027908496586162815
DNE 70% ***** 9.279847462054201e-05
DNE 80% ***** 3.4960701309014304e-05
DNE 90% ***** 0.00014062919265086962
Built Environment & Design
DNE 60% ***** 0.1440512484456709
DNE 70% ***** 0.009435338542806966
DNE 80% ***** 0.028472412755124545
DNE 90% ***** 0.026156611380006765
Clinical Medicine
DNE 60% ***** 0.006274155879010838
DNE 70% ***** 0.012711802317082898
DNE 80% ***** 0.020167622516480316
DNE 90% ***** 0.07265769425057962
Earth & Environmental Sciences
DNE 60% ***** 0.0006595511905788828
DNE 70% ***** 0.0008245818208916544
DNE 80% ***** 0.009280439891181474
DNE 90% ***** 0.005178063771911943
Economics & Business
DNE 60% ***** 0.028591613732577997
DNE 70% ***** 0.0010578362390259604
DNE 80% ***** 0.004163995481470227
DNE 90% ***** 0.0033211724573644842
Engineering
DNE

In [28]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 0.0003773478919103644
DNE 70% ***** 8.914838928443509e-05
DNE 80% ***** 0.00026000666334005884
DNE 90% ***** 0.0007445450632187914
Europe
DNE 60% ***** 0.05186463733721316
DNE 70% ***** 0.017499260138981808
DNE 80% ***** 0.0030556045512759875
DNE 90% ***** 0.00876566582558922
Middle East
DNE 60% ***** 0.04311590560816902
DNE 70% ***** 0.008308065023896591
DNE 80% ***** 0.00042468905694526286
DNE 90% ***** 0.017317039688540457
North Africa
DNE 60% ***** 0.11859096604891295
DNE 70% ***** 0.00019170269085967432
DNE 80% ***** 0.001138662012341646
DNE 90% ***** 0.003324235245681972
North America
DNE 60% ***** 0.04288205250660929
DNE 70% ***** 0.02298125551236851
DNE 80% ***** 0.009025908920616343
DNE 90% ***** 0.002349807916753109
Oceanic
DNE 60% ***** 0.03071688309436626
DNE 70% ***** 0.00023946021699105825
DNE 80% ***** 0.005549577106798381
DNE 90% ***** 0.0010544585915922234
South/Central America
DNE 60% ***** 0.007641392330863131
DNE 70% ***** 0.001911